## Webpage Extraction and Embedding (PolyU SAO)

### 1. Extracting raw text data

In [1]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [2]:
sao_URL = "https://www.polyu.edu.hk/sao/"
docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "html.parser")   # Strip the html syntax
    
    # Get the title of the page
    title = soup.title.string if soup.title else "No Title"

    # Attempt to find main content areas
    main_content = soup.find('main') or soup.find('article') or soup.find('div', class_='content')
    if main_content:
        text = re.sub(r"\n\n+", "\n\n", main_content.get_text()).strip()
    else:
        for element in soup.find_all(['header', 'footer', 'nav']):
            element.decompose()
        text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess newlines
    
    return f"{title}\n\n{text}"

sao_loader = RecursiveUrlLoader(
    max_depth=6,
    url=sao_URL,
    base_url=sao_URL,
    prevent_outside=True,
    exclude_dirs=[
        sao_URL+"News-and-Events", 
        sao_URL+"news-and-events",
        sao_URL+"Sitemap",
        sao_URL+"sitemap",
        sao_URL+"Search-Result",
        sao_URL+"search-result",
        sao_URL+"Personal-Information-Collection-Statement",
        sao_URL+"docdrive",
        sao_URL+"-",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = sao_loader.lazy_load()
for doc in docs_lazy:
    print(doc.metadata.get('source'))
    for key in unwanted_metadata:
        if key in doc.metadata:
            del doc.metadata[key]
    docs.append(doc)

https://www.polyu.edu.hk/sao/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/student-counselling/mental-health-educational-material-or-resources/academic/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/programmes-and-activities/wellness-ally/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/sports-facilities/charges-regulations/
https://www.polyu.edu.hk/sao/student-resources-and-support-section/residential-life/resources/
https://www.polyu.edu.hk/sao/student-development-section/holistic-student-development/on-campus-intercultural-development-programmes/happy-hour-gathering/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/student-counselling/mental-health-educational-material-or-resources/career/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/sports-development/sports-team/sports-scholarships-for-student-athletes-in-polyu/
https://www.polyu.edu.hk/sao/student-resources-and-support-section/special-needs-support/
http

In [3]:
idx = 76
print(f"Extracted number of webpages in SAO: {len(docs)}")
print(docs[idx].page_content[:])
print(docs[idx].metadata.get('source'))

'''
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in SAO: 292
Event Details | Student Affairs Office

PolyU Asian Universities Water Polo Invitational Tournament

PolyU Asian Universities Water Polo Invitational Tournament

 

Event Details
 

 
Pool Game:     
 4-6 July 2023

 
Gold & Bronze Medal matches:
 8 July 2023 

 
Venue :   
 Michael Clinton Swimming Pool, PolyU

 
Livestream link:   
 https://youtu.be/iM7kExbhR5U (For the Gold & Bronze Medal matches only)

 
 

 

 
 


Access to Swimming Pool                                            
More information, please click here.

 
 


Participating Teams                                    

A total of six collegiate teams in the Asian countries/regions participate in the Tournament. (In alphabetical order)

Juntendo University, Japan

Korea National Sport University, South Korea

Niigata Sangyo University, Japan

South China Agricultural University, China

The Chinese University of Hong Kong, Hong Kong China

The Hong Kong Polytechnic University, Hon

"\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1) if re.search(r'/([^/]+)/?$', source) else "unknown"
    chunk.metadata["chunk_id"] = f"PolyU_SAO_{url_end}_chunk_{i}"

    chunk.page_content = f"--- PolyU SAO Website URL: {source} ---\n\n{chunk.page_content}"

print("Number of chunks:", len(chunks))

Number of chunks: 1136


In [5]:
print(chunks[10])

page_content='--- PolyU SAO Website URL: https://www.polyu.edu.hk/sao/counselling-and-wellness-section/programmes-and-activities/wellness-ally/ ---

Trainings - Personal Enhancement                                        

Equipping Wellness Ally with mental health knowledge and helping skills

 
I gained a lot of mental health first-aid knowledge and learnt about the wellness concept.

 

 
I feel satisfied when I could receive the training because the knowledge helps me deal with my mental issues and also help my friends in need.

 

 
It is fruitful to be able to enhance my skills through the Mental Health First Aid programme.

 


Monthly Gatherings - Team Building                                        

Promoting personal growth and building relationships 

 
Knowing a group of people whose goal is also to pursue mental health promotion on campus.

 

 
I joined the monthly gathering which was a relaxing experience for me during the hard time.

 

 
It was satisfying to meet peop

### 3. Document Embedding in Chroma

In [6]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "sao_documents" if not SINGLE else "vaa_documents"

In [ ]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

C:\Users\Marcus\AppData\Local\Temp\ipykernel_2364\2527254279.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding_function = OllamaEmbeddings(model="bge-m3:567m", num_gpu=1) # Please OPEN Ollama first!!


0

In [8]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

In [9]:
for i, chunk in enumerate(chunks):
    print(f"Adding chunk {i+1}/{len(chunks)} to ChromaDB. Metadata: {chunk.metadata}\n")
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Adding chunk 1/1136 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affairs Office', 'chunk_id': 'PolyU_SAO_sao_chunk_0'}

Adding chunk 2/1136 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affairs Office', 'chunk_id': 'PolyU_SAO_sao_chunk_1'}

Adding chunk 3/1136 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affairs Office', 'chunk_id': 'PolyU_SAO_sao_chunk_2'}

Adding chunk 4/1136 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affairs Office', 'chunk_id': 'PolyU_SAO_sao_chunk_3'}

Adding chunk 5/1136 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student 

### 4. Simple Testing

In [10]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "How can I apply for residential hall in PolyU?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

C:\Users\Marcus\AppData\Local\Temp\ipykernel_2364\954563891.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


Content: --- PolyU SAO Website URL: https://www.polyu.edu.hk/sao/student-resources-and-support-section/residential-life/hall-admission/admission-policies-undergraduates/ ---

5. Residential Colleges
The Residential Colleges (RCs) may have additional admission criteria, such as requirements related to academic performance and student conduct. In addition to the standard Hall Admission Policies outlined here, applicants must also meet any specific admission criteria set by the respective RC, where applicable.
6. Applicants with exceptional needs
Applicants having exceptional needs (e.g. those students with physical disability or personal hardships) for Hall residence will be given special consideration. The applicants should send documentary proof to establish their special need or in support of their application by email to student.halls@polyu.edu.hk on or before 30 June every year for the following residential year. Late applications without good justifications may not be considered. A